In [6]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.stats.proportion as sp

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Analyzing category churn rates with confidence intervals

In [2]:

# ============================================================
# DATASET PATH
#
# Use the demographic dataset created from Listing 10.1.
# Change the filename here if you saved it under another name.
# ============================================================

data_set_path = str(
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset_demographic.csv"
)


In [3]:
# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected numeric churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE CATEGORY DATA
# ============================================================

def prepare_category_data(
    data_set_path,
    cat_col
):

    if not os.path.isfile(data_set_path):
        raise FileNotFoundError(
            f'"{data_set_path}" is not a valid dataset path.'
        )

    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    if cat_col not in churn_data.columns:
        raise ValueError(
            f'Category column "{cat_col}" not found.\n'
            f"Available columns:\n"
            f"{list(churn_data.columns)}"
        )

    if "is_churn" not in churn_data.columns:
        raise ValueError(
            '"is_churn" column is missing.'
        )

    churn_data = churn_data.copy()

    # Fill missing category values safely
    churn_data[cat_col] = (
        churn_data[cat_col]
        .fillna("-na-")
        .astype(str)
    )

    # Convert target to 0 / 1
    churn_data["is_churn"] = (
        convert_churn_target(
            churn_data["is_churn"]
        )
        .astype(int)
    )

    return churn_data


# ============================================================
# CATEGORY CHURN SUMMARY
# ============================================================

def category_churn_summary(
    churn_data,
    cat_col,
    data_set_path
):

    summary = (
        churn_data
        .groupby(
            cat_col,
            dropna=False
        )
        .agg(
            n_accounts=(
                cat_col,
                "size"
            ),
            n_churn=(
                "is_churn",
                "sum"
            ),
            churn_rate=(
                "is_churn",
                "mean"
            )
        )
    )

    # ========================================================
    # WILSON CONFIDENCE INTERVALS
    # ========================================================

    lower_conf, upper_conf = (
        sp.proportion_confint(
            count=summary["n_churn"],
            nobs=summary["n_accounts"],
            alpha=0.05,
            method="wilson"
        )
    )

    summary["lo_conf"] = lower_conf
    summary["hi_conf"] = upper_conf

    # ========================================================
    # ERROR-BAR DISTANCES
    # ========================================================

    summary["lo_int"] = (
        summary["churn_rate"]
        - summary["lo_conf"]
    )

    summary["hi_int"] = (
        summary["hi_conf"]
        - summary["churn_rate"]
    )

    # ========================================================
    # SHARE OF DATASET
    # ========================================================

    summary[
        f"{cat_col}_percent"
    ] = (
        summary["n_accounts"]
        / len(churn_data)
    )

    # ========================================================
    # SORT
    # ========================================================

    summary = (
        summary
        .sort_values(
            "churn_rate",
            ascending=False
        )
    )

    # ========================================================
    # SAVE
    # ========================================================

    save_path = (
        data_set_path.replace(
            ".csv",
            f"_{cat_col}_churn_category.csv"
        )
    )

    summary.to_csv(
        save_path
    )

    print(
        f"Saving summary to:\n"
        f"{save_path}"
    )

    return summary


# ============================================================
# CATEGORY CHURN PLOT
# ============================================================

def category_churn_plot(
    cat_col,
    summary,
    data_set_path
):

    n_category = summary.shape[0]

    if n_category == 0:
        raise ValueError(
            "No category rows available to plot."
        )

    plt.figure(
        figsize=(
            max(
                6,
                0.8 * n_category
            ),
            4
        )
    )

    yerr = np.vstack(
        [
            summary["lo_int"].to_numpy(),
            summary["hi_int"].to_numpy()
        ]
    )

    plt.bar(
        x=summary.index.astype(str),
        height=summary["churn_rate"],
        yerr=yerr,
        capsize=max(
            2,
            min(
                8,
                80 / n_category
            )
        )
    )

    plt.xlabel(
        cat_col
    )

    plt.ylabel(
        "Category Churn Rate"
    )

    plt.title(
        f'Average Churn by "{cat_col}"'
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.grid(
        axis="y"
    )

    plt.tight_layout()

    save_path = (
        data_set_path.replace(
            ".csv",
            f"_{cat_col}_churn_category.png"
        )
    )

    plt.savefig(
        save_path
    )

    plt.close()

    print(
        f"Saving plot to:\n"
        f"{save_path}"
    )

    return save_path


# ============================================================
# LISTING 10.2 MAIN FUNCTION
# ============================================================

def category_churn_cohorts(
    data_set_path,
    cat_col
):

    churn_data = (
        prepare_category_data(
            data_set_path,
            cat_col
        )
    )

    summary = (
        category_churn_summary(
            churn_data,
            cat_col,
            data_set_path
        )
    )

    category_churn_plot(
        cat_col,
        summary,
        data_set_path
    )

    print(
        f"\nCategory analysis complete for: "
        f"{cat_col}"
    )

    display(
        summary
    )

    return summary


# ============================================================
# RUN EXAMPLE
#
# Use one demographic categorical field at a time.
# ============================================================

channel_summary = category_churn_cohorts(
    data_set_path=data_set_path,
    cat_col="channel"
)

Saving summary to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_channel_churn_category.csv
Saving plot to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_channel_churn_category.png

Category analysis complete for: channel


,n_accounts,n_churn,churn_rate,lo_conf,hi_conf,lo_int,hi_int,channel_percent
channel,,,,,,,,
web,4851,148,0.030509,0.026029,0.035733,0.004481,0.005224,0.147460
appstore1,9807,222,0.022637,0.019875,0.025773,0.002762,0.003136,0.298112
appstore2,18239,275,0.015078,0.013408,0.016951,0.001669,0.001873,0.554427


# “Grouped category cohort analysis”


In [4]:

# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE CATEGORY DATA
# ============================================================

def prepare_category_data(
    data_set_path,
    cat_col
):

    if not os.path.isfile(data_set_path):

        raise FileNotFoundError(
            f'"{data_set_path}" is not a valid dataset path.'
        )

    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    if cat_col not in churn_data.columns:

        raise ValueError(
            f'Category column "{cat_col}" not found.\n'
            f"Available columns:\n"
            f"{list(churn_data.columns)}"
        )

    if "is_churn" not in churn_data.columns:

        raise ValueError(
            '"is_churn" column is missing.'
        )

    churn_data = churn_data.copy()

    churn_data[cat_col] = (
        churn_data[cat_col]
        .fillna("-na-")
        .astype(str)
    )

    churn_data["is_churn"] = (
        convert_churn_target(
            churn_data["is_churn"]
        )
        .astype(int)
    )

    return churn_data


# ============================================================
# CATEGORY SUMMARY
# ============================================================

def category_churn_summary(
    churn_data,
    cat_col,
    data_set_path
):

    summary = (
        churn_data
        .groupby(
            cat_col,
            dropna=False
        )
        .agg(
            n_accounts=(
                cat_col,
                "size"
            ),
            n_churn=(
                "is_churn",
                "sum"
            ),
            churn_rate=(
                "is_churn",
                "mean"
            )
        )
    )

    lower_conf, upper_conf = (
        sp.proportion_confint(
            count=summary["n_churn"],
            nobs=summary["n_accounts"],
            alpha=0.05,
            method="wilson"
        )
    )

    summary["lo_conf"] = (
        lower_conf
    )

    summary["hi_conf"] = (
        upper_conf
    )

    summary["lo_int"] = (
        summary["churn_rate"]
        - summary["lo_conf"]
    )

    summary["hi_int"] = (
        summary["hi_conf"]
        - summary["churn_rate"]
    )

    summary[
        f"{cat_col}_percent"
    ] = (
        summary["n_accounts"]
        / len(churn_data)
    )

    summary = (
        summary
        .sort_values(
            "churn_rate",
            ascending=False
        )
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            f"_{cat_col}_churn_category.csv"
        )
    )

    summary.to_csv(
        save_path
    )

    print(
        f"Saved summary to:\n"
        f"{save_path}"
    )

    return summary


# ============================================================
# CATEGORY PLOT
# ============================================================

def category_churn_plot(
    cat_col,
    summary,
    data_set_path
):

    n_category = (
        summary.shape[0]
    )

    if n_category == 0:

        raise ValueError(
            "No categories available to plot."
        )

    plt.figure(
        figsize=(
            max(
                6,
                0.8 * n_category
            ),
            4
        )
    )

    yerr = np.vstack(
        [
            summary[
                "lo_int"
            ].to_numpy(),

            summary[
                "hi_int"
            ].to_numpy()
        ]
    )

    plt.bar(
        x=summary.index.astype(str),
        height=summary["churn_rate"],
        yerr=yerr,
        capsize=max(
            2,
            min(
                8,
                80 / n_category
            )
        )
    )

    plt.xlabel(
        cat_col
    )

    plt.ylabel(
        "Category Churn Rate"
    )

    plt.title(
        f'Average Churn by "{cat_col}"'
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.grid(
        axis="y"
    )

    plt.tight_layout()

    save_path = (
        data_set_path.replace(
            ".csv",
            f"_{cat_col}_churn_category.png"
        )
    )

    plt.savefig(
        save_path
    )

    plt.close()

    print(
        f"Saved plot to:\n"
        f"{save_path}"
    )


# ============================================================
# GROUP CATEGORY COLUMN
# ============================================================

def group_category_column(
    df,
    cat_col,
    group_dict
):

    if cat_col not in df.columns:

        raise ValueError(
            f'Column "{cat_col}" not found.'
        )

    # --------------------------------------------------------
    # Invert:
    #
    # {
    #     "Group A": ["x", "y"]
    # }
    #
    # into:
    #
    # {
    #     "x": "Group A",
    #     "y": "Group A"
    # }
    # --------------------------------------------------------

    group_lookup = {}

    for group_name, values in group_dict.items():

        for value in values:

            if value in group_lookup:

                raise ValueError(
                    f'Category "{value}" appears in '
                    "more than one group."
                )

            group_lookup[value] = (
                group_name
            )

    group_cat_col = (
        f"{cat_col}_group"
    )

    # Categories not explicitly grouped
    # retain their original value.
    df[group_cat_col] = (
        df[cat_col]
        .map(group_lookup)
        .fillna(df[cat_col])
    )

    df.drop(
        columns=[cat_col],
        inplace=True
    )

    return group_cat_col


# ============================================================
# LISTING 10.3 — GROUPED CATEGORY COHORTS
# ============================================================

def grouped_category_cohorts(
    data_set_path,
    cat_col,
    groups
):

    churn_data = (
        prepare_category_data(
            data_set_path,
            cat_col
        )
    )

    group_cat_col = (
        group_category_column(
            churn_data,
            cat_col,
            groups
        )
    )

    summary = (
        category_churn_summary(
            churn_data,
            group_cat_col,
            data_set_path
        )
    )

    category_churn_plot(
        group_cat_col,
        summary,
        data_set_path
    )

    print(
        f"\nGrouped category analysis complete: "
        f"{group_cat_col}"
    )

    display(
        summary
    )

    return summary


# ============================================================
# EXAMPLE 1 — GROUP COUNTRIES
# ============================================================

country_groups = {

    "North America": [
        "US",
        "CA"
    ],

    "Western Europe": [
        "GB",
        "DE",
        "FR"
    ],

    "Asia": [
        "JP"
    ]
}


country_group_summary = (
    grouped_category_cohorts(
        data_set_path=data_set_path,
        cat_col="country",
        groups=country_groups
    )
)

Saved summary to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_country_group_churn_category.csv
Saved plot to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_country_group_churn_category.png

Grouped category analysis complete: country_group


,n_accounts,n_churn,churn_rate,lo_conf,hi_conf,lo_int,hi_int,country_group_percent
country_group,,,,,,,,
PT,259,8,0.030888,0.015733,0.059755,0.015155,0.028867,0.007873
IT,312,8,0.025641,0.013049,0.049772,0.012592,0.024131,0.009484
North America,6220,150,0.024116,0.020587,0.028232,0.003529,0.004116,0.189075
CO,711,16,0.022504,0.013898,0.036241,0.008605,0.013737,0.021613
ID,321,7,0.021807,0.010603,0.044321,0.011204,0.022514,0.009758
NL,323,7,0.021672,0.010537,0.044051,0.011135,0.022379,0.009819
IN,1341,29,0.021626,0.015099,0.030885,0.006527,0.009260,0.040764
MX,1263,26,0.020586,0.014087,0.029993,0.006499,0.009407,0.038393
-na-,3716,74,0.019914,0.015893,0.024926,0.004021,0.005013,0.112959


#  Creating dummy variables

In [5]:
# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    # Already boolean
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)

    # Numeric 0 / 1
    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected numeric churn values: {values}"
            )

        return (
            series
            .astype(int)
        )

    # String values
    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": 1,
        "false": 0,
        "t": 1,
        "f": 0,
        "1": 1,
        "0": 0,
        "yes": 1,
        "no": 0
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(int)
    )


# ============================================================
# GROUP CATEGORY COLUMN
# ============================================================

def group_category_column(
    df,
    cat_col,
    group_dict
):

    if cat_col not in df.columns:

        raise ValueError(
            f'Column "{cat_col}" not found.'
        )

    group_lookup = {}

    for group_name, values in group_dict.items():

        for value in values:

            if value in group_lookup:

                raise ValueError(
                    f'Category "{value}" appears '
                    "in more than one group."
                )

            group_lookup[value] = group_name

    group_cat_col = (
        f"{cat_col}_group"
    )

    original_values = (
        df[cat_col]
        .fillna("-na-")
        .astype(str)
    )

    df[group_cat_col] = (
        original_values
        .map(group_lookup)
        .fillna(original_values)
    )

    df.drop(
        columns=[cat_col],
        inplace=True
    )

    return group_cat_col


# ============================================================
# LISTING 10.4 — CREATE DUMMY VARIABLES
# ============================================================

def dummy_variables(
    data_set_path,
    groups=None,
    current=False
):

    if groups is None:
        groups = {}

    if not os.path.isfile(
        data_set_path
    ):

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{data_set_path}"
        )


    # ========================================================
    # LOAD DATA
    # ========================================================

    raw_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    print(
        f"Loaded rows: {len(raw_data):,}"
    )

    print(
        f"Original columns: {raw_data.shape[1]}"
    )


    # ========================================================
    # PRESERVE / CONVERT TARGET
    # ========================================================

    if not current:

        if "is_churn" not in raw_data.columns:

            raise ValueError(
                "Training dataset must contain "
                "'is_churn'."
            )

        raw_data["is_churn"] = (
            convert_churn_target(
                raw_data["is_churn"]
            )
        )

        print(
            "\nChurn target:"
        )

        print(
            raw_data[
                "is_churn"
            ].value_counts(
                dropna=False
            )
        )


    # ========================================================
    # GROUP REQUESTED CATEGORIES
    # ========================================================

    for (
        cat_col,
        group_dict
    ) in groups.items():

        if cat_col not in raw_data.columns:

            raise ValueError(
                f'Grouping column "{cat_col}" '
                "is missing."
            )

        group_category_column(
            raw_data,
            cat_col,
            group_dict
        )


    # ========================================================
    # IDENTIFY CATEGORICAL FEATURES
    #
    # CRITICAL:
    # is_churn must NOT be converted into dummies.
    # ========================================================

    categorical_columns = (
        raw_data
        .select_dtypes(
            include=[
                "object",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        col
        for col in categorical_columns
        if col != "is_churn"
    ]

    print(
        "\nCategorical feature columns:"
    )

    print(
        categorical_columns
    )


    # ========================================================
    # SAVE ORIGINAL CATEGORICAL COLUMN NAMES
    # ========================================================

    original_categorical_columns = (
        categorical_columns.copy()
    )


    # ========================================================
    # CREATE DUMMY VARIABLES
    # ========================================================

    data_w_dummies = pd.get_dummies(
        raw_data,
        columns=categorical_columns,
        dummy_na=True,
        dtype=int
    )


    # ========================================================
    # VERIFY TARGET SURVIVED
    # ========================================================

    if not current:

        if "is_churn" not in data_w_dummies.columns:

            raise ValueError(
                "is_churn disappeared during "
                "dummy-variable creation."
            )

        unique_target_values = set(
            data_w_dummies[
                "is_churn"
            ].unique()
        )

        if not unique_target_values.issubset(
            {0, 1}
        ):

            raise ValueError(
                "is_churn is not binary after "
                "dummy-variable creation."
            )


    # ========================================================
    # FIND NEW DUMMY COLUMNS
    # ========================================================

    original_columns_after_grouping = set(
        raw_data.columns
    )

    dummy_columns = sorted(
        [
            col
            for col in data_w_dummies.columns
            if col not in original_columns_after_grouping
        ]
    )


    print(
        f"\nDummy variables created: "
        f"{len(dummy_columns)}"
    )

    print(
        "\nDummy columns:"
    )

    for col in dummy_columns:

        print(
            f"  - {col}"
        )


    # ========================================================
    # SAVE FULL XGBOOST DATASET
    # ========================================================

    xgb_path = (
        data_set_path.replace(
            ".csv",
            "_xgbdummies.csv"
        )
    )

    data_w_dummies.to_csv(
        xgb_path
    )

    print(
        f"\nSaved full dummy dataset to:\n"
        f"{xgb_path}"
    )


    # ========================================================
    # SAVE DUMMY COLUMN DEFINITIONS
    # ========================================================

    dummy_col_df = pd.DataFrame(
        {
            "metrics":
                dummy_columns
        },
        index=dummy_columns
    )

    dummy_groupmets_path = (
        data_set_path.replace(
            ".csv",
            "_dummies_groupmets.csv"
        )
    )

    dummy_col_df.to_csv(
        dummy_groupmets_path
    )

    print(
        f"\nSaved dummy column definitions to:\n"
        f"{dummy_groupmets_path}"
    )


    # ========================================================
    # DUMMY-ONLY DATASET
    # ========================================================

    dummy_score_columns = (
        dummy_columns.copy()
    )

    if not current:

        dummy_score_columns.append(
            "is_churn"
        )

    dummies_only = (
        data_w_dummies[
            dummy_score_columns
        ]
        .copy()
    )

    dummy_groupscore_path = (
        data_set_path.replace(
            ".csv",
            "_dummies_groupscore.csv"
        )
    )

    dummies_only.to_csv(
        dummy_groupscore_path
    )

    print(
        f"\nSaved dummy-only dataset to:\n"
        f"{dummy_groupscore_path}"
    )


    # ========================================================
    # SAVE DATASET WITHOUT STRING CATEGORIES
    #
    # Keep numeric metrics + target.
    # ========================================================

    no_category_data = (
        raw_data.drop(
            columns=original_categorical_columns,
            errors="ignore"
        )
        .copy()
    )

    nocat_path = (
        data_set_path.replace(
            ".csv",
            "_nocat.csv"
        )
    )

    no_category_data.to_csv(
        nocat_path
    )

    print(
        f"\nSaved no-category dataset to:\n"
        f"{nocat_path}"
    )


    # ========================================================
    # FINAL QA
    # ========================================================

    print(
        "\nFinal QA"
    )

    print(
        "---------"
    )

    print(
        f"Rows: {len(data_w_dummies):,}"
    )

    print(
        f"Full dummy dataset columns: "
        f"{data_w_dummies.shape[1]}"
    )

    print(
        f"Dummy-only columns: "
        f"{dummies_only.shape[1]}"
    )

    if not current:

        print(
            f"Churns: "
            f"{int(data_w_dummies['is_churn'].sum()):,}"
        )

        print(
            f"Non-churns: "
            f"{int((data_w_dummies['is_churn'] == 0).sum()):,}"
        )

        print(
            f"Churn rate: "
            f"{data_w_dummies['is_churn'].mean():.3%}"
        )

    print(
        "\nListing 10.4 complete."
    )

    return (
        data_w_dummies,
        dummies_only,
        no_category_data,
        dummy_col_df
    )


# ============================================================
# COUNTRY GROUPS
# ============================================================

groups = {

    "country": {

        "North America": [
            "US",
            "CA"
        ],

        "Western Europe": [
            "GB",
            "DE",
            "FR"
        ],

        "Asia": [
            "JP"
        ]
    }
}


# ============================================================
# RUN LISTING 10.4
# ============================================================

(
    demographic_xgb_data,
    demographic_dummy_scores,
    demographic_nocat_data,
    dummy_columns
) = dummy_variables(
    data_set_path=data_set_path,
    groups=groups,
    current=False
)


# ============================================================
# DISPLAY OUTPUTS
# ============================================================

print(
    "\nDummy variable definitions:"
)

display(
    dummy_columns
)


print(
    "\nDummy-only training dataset:"
)

display(
    demographic_dummy_scores.head()
)


print(
    "\nFull XGBoost dummy dataset:"
)

display(
    demographic_xgb_data.head()
)

Loaded rows: 32,897
Original columns: 21

Churn target:
is_churn
0    32252
1      645
Name: count, dtype: int64

Categorical feature columns:
['channel', 'country_group']

Dummy variables created: 29

Dummy columns:
  - channel_appstore1
  - channel_appstore2
  - channel_nan
  - channel_web
  - country_group_-na-
  - country_group_AR
  - country_group_AU
  - country_group_Asia
  - country_group_BR
  - country_group_CH
  - country_group_CN
  - country_group_CO
  - country_group_DK
  - country_group_ES
  - country_group_GR
  - country_group_ID
  - country_group_IN
  - country_group_IT
  - country_group_KR
  - country_group_MX
  - country_group_NL
  - country_group_NO
  - country_group_NZ
  - country_group_North America
  - country_group_PT
  - country_group_RU
  - country_group_SE
  - country_group_Western Europe
  - country_group_nan

Saved full dummy dataset to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_xgbdummies.csv

Saved dummy column definitions 

,metrics
channel_appstore1,channel_appstore1
channel_appstore2,channel_appstore2
channel_nan,channel_nan
channel_web,channel_web
country_group_-na-,country_group_-na-
country_group_AR,country_group_AR
country_group_AU,country_group_AU
country_group_Asia,country_group_Asia
country_group_BR,country_group_BR
country_group_CH,country_group_CH



Dummy-only training dataset:


,,channel_appstore1,channel_appstore2,channel_nan,channel_web,country_group_-na-,country_group_AR,country_group_AU,country_group_Asia,country_group_BR,country_group_CH,...,country_group_NL,country_group_NO,country_group_NZ,country_group_North America,country_group_PT,country_group_RU,country_group_SE,country_group_Western Europe,country_group_nan,is_churn
account_id,observation_date,,,,,,,,,,,,,,,,,,,,,
82,2020-02-09,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
122,2020-02-09,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
184,2020-02-09,0,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
262,2020-02-09,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
291,2020-02-09,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0



Full XGBoost dummy dataset:


,,is_churn,customer_age,like_per_month,newfriend_per_month,post_per_month,adview_per_month,dislike_per_month,unfriend_per_month,message_per_month,reply_per_month,...,country_group_MX,country_group_NL,country_group_NO,country_group_NZ,country_group_North America,country_group_PT,country_group_RU,country_group_SE,country_group_Western Europe,country_group_nan
account_id,observation_date,,,,,,,,,,,,,,,,,,,,,
82,2020-02-09,0,18.739726,101,8,161,422,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
122,2020-02-09,0,79.917808,8,3,3,8,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
184,2020-02-09,0,66.421918,12,4,5,19,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
262,2020-02-09,0,19.871233,22,6,8,19,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
291,2020-02-09,0,76.983562,18,3,0,11,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


# “ Merging dummy variables with grouped metric scores”

In [9]:
# ============================================================
# DATASET PATH
# ============================================================

data_set_path = str(
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset_demographic_nocat.csv"
)

# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected churn values: {values}"
            )

        return series.astype(int)

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": 1,
        "false": 0,
        "t": 1,
        "f": 0,
        "1": 1,
        "0": 0,
        "yes": 1,
        "no": 0
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(int)
    )


# ============================================================
# SUMMARY STATISTICS
# ============================================================

def dataset_stats(
    data_set_path
):

    data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    if "is_churn" in data.columns:
        data["is_churn"] = convert_churn_target(
            data["is_churn"]
        )

    numeric_data = data.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if numeric_data.isna().any().any():

        bad_cols = numeric_data.columns[
            numeric_data.isna().any()
        ].tolist()

        raise ValueError(
            f"Non-numeric/NaN columns found: {bad_cols}"
        )

    stats = (
        numeric_data
        .describe()
        .T
    )

    stats["skew"] = (
        numeric_data.skew()
    )

    stats["1%"] = (
        numeric_data.quantile(
            0.01
        )
    )

    stats["99%"] = (
        numeric_data.quantile(
            0.99
        )
    )

    stats["nonzero"] = (
        numeric_data.ne(0).sum()
        / len(numeric_data)
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            "_summarystats.csv"
        )
    )

    stats.to_csv(
        save_path
    )

    print(
        f"Saved:\n{save_path}"
    )

    return stats


# ============================================================
# TRANSFORMS
# ============================================================

def transform_skew_columns(
    data,
    columns
):

    for col in columns:
        data[col] = np.log1p(
            data[col]
        )


def transform_fattail_columns(
    data,
    columns
):

    for col in columns:
        data[col] = np.arcsinh(
            data[col]
        )


# ============================================================
# SCORE METRICS
# ============================================================

def fat_tail_scores(
    data_set_path,
    skew_thresh=4.0
):

    data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    if "is_churn" not in data.columns:
        raise ValueError(
            "is_churn column missing."
        )

    target = convert_churn_target(
        data["is_churn"]
    )

    scores = (
        data.drop(
            columns=["is_churn"]
        )
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    if scores.isna().any().any():

        bad_cols = scores.columns[
            scores.isna().any()
        ].tolist()

        raise ValueError(
            f"NaN/non-numeric features found: {bad_cols}"
        )

    stat_path = (
        data_set_path.replace(
            ".csv",
            "_summarystats.csv"
        )
    )

    stats = pd.read_csv(
        stat_path,
        index_col=0
    )

    stats = stats.drop(
        index="is_churn",
        errors="ignore"
    )

    common = [
        col
        for col in scores.columns
        if col in stats.index
    ]

    scores = scores[
        common
    ]

    stats = stats.loc[
        common
    ]

    skewed_columns = (
        (stats["skew"] > skew_thresh)
        &
        (stats["min"] >= 0)
    )

    fattail_columns = (
        (stats["skew"].abs() > skew_thresh)
        &
        (stats["min"] < 0)
    )

    transform_skew_columns(
        scores,
        skewed_columns[
            skewed_columns
        ].index
    )

    transform_fattail_columns(
        scores,
        fattail_columns[
            fattail_columns
        ].index
    )

    mean_vals = (
        scores.mean()
    )

    std_vals = (
        scores.std()
    )

    valid_cols = std_vals[
        std_vals > 0
    ].index

    scores = scores[
        valid_cols
    ]

    mean_vals = mean_vals[
        valid_cols
    ]

    std_vals = std_vals[
        valid_cols
    ]

    scores = (
        scores - mean_vals
    ) / std_vals

    scores["is_churn"] = (
        target
    )

    score_path = (
        data_set_path.replace(
            ".csv",
            "_scores.csv"
        )
    )

    scores.to_csv(
        score_path
    )

    print(
        f"Saved:\n{score_path}"
    )

    param_df = pd.DataFrame(
        index=valid_cols
    )

    param_df["skew_score"] = (
        skewed_columns
        .reindex(
            valid_cols
        )
        .fillna(False)
    )

    param_df["fattail_score"] = (
        fattail_columns
        .reindex(
            valid_cols
        )
        .fillna(False)
    )

    param_df["mean"] = (
        mean_vals
    )

    param_df["std"] = (
        std_vals
    )

    param_path = (
        data_set_path.replace(
            ".csv",
            "_score_params.csv"
        )
    )

    param_df.to_csv(
        param_path
    )

    print(
        f"Saved:\n{param_path}"
    )

    return scores


# ============================================================
# FIND METRIC GROUPS
# ============================================================

def find_metric_groups(
    data_set_path,
    group_corr_thresh=0.55
):

    score_path = (
        data_set_path.replace(
            ".csv",
            "_scores.csv"
        )
    )

    scores = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    metrics = scores.drop(
        columns=["is_churn"],
        errors="ignore"
    )

    corr = (
        metrics.corr()
        .fillna(0)
    )

    np.fill_diagonal(
        corr.values,
        1.0
    )

    dissimilarity = (
        1.0 - corr
    )

    dissimilarity = (
        dissimilarity
        + dissimilarity.T
    ) / 2.0

    np.fill_diagonal(
        dissimilarity.values,
        0.0
    )

    condensed = squareform(
        dissimilarity.values,
        checks=False
    )

    linkage_matrix = linkage(
        condensed,
        method="single"
    )

    cluster_ids = fcluster(
        linkage_matrix,
        t=1.0 - group_corr_thresh,
        criterion="distance"
    )

    cluster_series = pd.Series(
        cluster_ids,
        index=metrics.columns,
        name="group"
    )

    group_sizes = (
        cluster_series
        .value_counts()
        .sort_values(
            ascending=False
        )
    )

    relabel = {
        old_group: new_group
        for new_group, old_group
        in enumerate(
            group_sizes.index,
            start=1
        )
    }

    cluster_series = (
        cluster_series.map(
            relabel
        )
    )

    groups = sorted(
        cluster_series.unique()
    )

    load_mat = pd.DataFrame(
        0.0,
        index=metrics.columns,
        columns=[
            f"group_{g}"
            for g in groups
        ]
    )

    for group in groups:

        members = cluster_series[
            cluster_series == group
        ].index

        group_size = len(
            members
        )

        if group_size == 1:
            weight = 1.0

        else:
            weight = (
                1.0
                /
                (
                    np.sqrt(
                        group_corr_thresh
                    )
                    * group_size
                )
            )

        load_mat.loc[
            members,
            f"group_{group}"
        ] = weight

    load_path = (
        data_set_path.replace(
            ".csv",
            "_load_mat.csv"
        )
    )

    load_mat.to_csv(
        load_path
    )

    print(
        f"Saved:\n{load_path}"
    )

    group_df = (
        cluster_series
        .rename("group")
        .to_frame()
    )

    group_path = (
        data_set_path.replace(
            ".csv",
            "_groupmets.csv"
        )
    )

    group_df.to_csv(
        group_path
    )

    print(
        f"Saved:\n{group_path}"
    )

    return (
        load_mat,
        group_df
    )


# ============================================================
# APPLY METRIC GROUPS
# ============================================================

def apply_metric_groups(
    data_set_path
):

    score_path = (
        data_set_path.replace(
            ".csv",
            "_scores.csv"
        )
    )

    load_path = (
        data_set_path.replace(
            ".csv",
            "_load_mat.csv"
        )
    )

    scores = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    load_mat = pd.read_csv(
        load_path,
        index_col=0
    )

    target = (
        scores["is_churn"]
        .copy()
    )

    metric_scores = scores.drop(
        columns=["is_churn"]
    )

    metric_scores = metric_scores[
        load_mat.index
    ]

    grouped_values = (
        metric_scores.to_numpy()
        @
        load_mat.to_numpy()
    )

    grouped_scores = pd.DataFrame(
        grouped_values,
        index=metric_scores.index,
        columns=load_mat.columns
    )

    grouped_scores["is_churn"] = (
        target
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            "_groupscore.csv"
        )
    )

    grouped_scores.to_csv(
        save_path
    )

    print(
        f"Saved:\n{save_path}"
    )

    return grouped_scores


# ============================================================
# RUN COMPLETE NOCAT PIPELINE
# ============================================================

print(
    "\n1. Summary statistics"
)

dataset_stats(
    data_set_path
)

print(
    "\n2. Metric scoring"
)

fat_tail_scores(
    data_set_path,
    skew_thresh=4.0
)

print(
    "\n3. Metric grouping"
)

find_metric_groups(
    data_set_path,
    group_corr_thresh=0.55
)

print(
    "\n4. Apply grouped metrics"
)

nocat_groupscore = (
    apply_metric_groups(
        data_set_path
    )
)

print(
    "\nNo-category preparation complete."
)

display(
    nocat_groupscore.head()
)


1. Summary statistics
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_summarystats.csv

2. Metric scoring
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_scores.csv
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_score_params.csv

3. Metric grouping
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_load_mat.csv
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_groupmets.csv

4. Apply grouped metrics
Saved:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_nocat_groupscore.csv

No-category preparation complete.


,,group_1,group_2,group_3,group_4,is_churn
account_id,observation_date,,,,,
82,2020-02-09,2.230020,-1.395082,0.223902,-1.235197,0
122,2020-02-09,-1.492277,1.633194,-0.418416,-1.235197,0
184,2020-02-09,-0.906755,0.965156,-0.289952,-1.235197,0
262,2020-02-09,-0.547187,-1.339073,-0.033025,-1.235197,0
291,2020-02-09,-1.641782,1.487950,-0.418416,-1.235197,0


# “Preparing a current customer dataset with demographic fields”


In [10]:
# ============================================================
# BASE HISTORICAL DATASET
# ============================================================

data_set_path = str(
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset_demographic.csv"
)


# ============================================================
# HELPER — GROUP CATEGORY COLUMN
# ============================================================

def group_category_column(
    df,
    cat_col,
    group_dict
):

    if cat_col not in df.columns:
        raise ValueError(
            f'Column "{cat_col}" not found.'
        )

    group_lookup = {}

    for group_name, values in group_dict.items():

        for value in values:

            if value in group_lookup:
                raise ValueError(
                    f'Category "{value}" appears '
                    "in more than one group."
                )

            group_lookup[value] = group_name

    group_cat_col = (
        f"{cat_col}_group"
    )

    original_values = (
        df[cat_col]
        .fillna("-na-")
        .astype(str)
    )

    df[group_cat_col] = (
        original_values
        .map(group_lookup)
        .fillna(original_values)
    )

    df.drop(
        columns=[cat_col],
        inplace=True
    )

    return group_cat_col


# ============================================================
# CREATE DUMMIES FOR CURRENT DATA
# ============================================================

def dummy_variables(
    current_path,
    groups=None
):

    if groups is None:
        groups = {}

    if not os.path.isfile(
        current_path
    ):
        raise FileNotFoundError(
            f"Current dataset not found:\n"
            f"{current_path}"
        )

    raw_data = pd.read_csv(
        current_path,
        index_col=[0, 1]
    )

    print(
        f"Loaded current rows: "
        f"{len(raw_data):,}"
    )

    # --------------------------------------------------------
    # GROUP SELECTED CATEGORICAL VARIABLES
    # --------------------------------------------------------

    for cat_col, group_dict in groups.items():

        if cat_col not in raw_data.columns:
            raise ValueError(
                f'Grouping column "{cat_col}" '
                "is missing from current data."
            )

        group_category_column(
            raw_data,
            cat_col,
            group_dict
        )

    # --------------------------------------------------------
    # IDENTIFY CATEGORICAL COLUMNS
    # --------------------------------------------------------

    cat_cols = (
        raw_data
        .select_dtypes(
            include=[
                "object",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    # --------------------------------------------------------
    # CREATE DUMMY VARIABLES
    # --------------------------------------------------------

    data_w_dummies = pd.get_dummies(
        raw_data,
        columns=cat_cols,
        dummy_na=True,
        dtype=int
    )

    raw_columns = set(
        raw_data.columns
    )

    dummy_columns = sorted(
        [
            col
            for col in data_w_dummies.columns
            if col not in raw_columns
        ]
    )

    # --------------------------------------------------------
    # SAVE FULL CURRENT DUMMY DATA
    # --------------------------------------------------------

    xgb_path = current_path.replace(
        ".csv",
        "_xgbdummies.csv"
    )

    data_w_dummies.to_csv(
        xgb_path
    )

    # --------------------------------------------------------
    # SAVE CURRENT DUMMY LIST
    # --------------------------------------------------------

    dummy_col_df = pd.DataFrame(
        {
            "metrics":
                dummy_columns
        },
        index=dummy_columns
    )

    dummy_groupmets_path = (
        current_path.replace(
            ".csv",
            "_dummies_groupmets.csv"
        )
    )

    dummy_col_df.to_csv(
        dummy_groupmets_path
    )

    # --------------------------------------------------------
    # SAVE CURRENT DUMMIES ONLY
    # --------------------------------------------------------

    current_dummies = (
        data_w_dummies[
            dummy_columns
        ]
        .copy()
    )

    dummy_groupscore_path = (
        current_path.replace(
            ".csv",
            "_dummies_groupscore.csv"
        )
    )

    current_dummies.to_csv(
        dummy_groupscore_path
    )

    # --------------------------------------------------------
    # SAVE CURRENT DATA WITHOUT CATEGORIES
    # --------------------------------------------------------

    current_nocat = (
        raw_data.drop(
            columns=cat_cols,
            errors="ignore"
        )
        .copy()
    )

    nocat_path = (
        current_path.replace(
            ".csv",
            "_nocat.csv"
        )
    )

    current_nocat.to_csv(
        nocat_path
    )

    print(
        f"Saved current dummy dataset:\n"
        f"{dummy_groupscore_path}"
    )

    print(
        f"Saved current no-category dataset:\n"
        f"{nocat_path}"
    )

    print(
        f"Current dummy columns: "
        f"{len(dummy_columns)}"
    )

    return (
        current_dummies,
        current_nocat,
        dummy_col_df
    )


# ============================================================
# ALIGN CURRENT DUMMIES WITH HISTORICAL TRAINING DUMMIES
# ============================================================

def align_dummies(
    current_dummies,
    data_set_path
):

    historical_groupmets_path = (
        data_set_path.replace(
            ".csv",
            "_dummies_groupmets.csv"
        )
    )

    if not os.path.isfile(
        historical_groupmets_path
    ):
        raise FileNotFoundError(
            "Historical dummy definition file "
            "not found.\n"
            "Run Listing 10.4 on the historical "
            "dataset first.\n"
            f"Expected:\n"
            f"{historical_groupmets_path}"
        )

    historical_groupmets = pd.read_csv(
        historical_groupmets_path,
        index_col=0
    )

    if "metrics" not in historical_groupmets.columns:
        raise ValueError(
            "Historical dummy definition file "
            "must contain a 'metrics' column."
        )

    old_dummies = (
        historical_groupmets[
            "metrics"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    old_dummy_set = set(
        old_dummies
    )

    new_dummy_set = set(
        current_dummies.columns
    )

    # --------------------------------------------------------
    # HISTORICAL CATEGORY ABSENT FROM CURRENT DATA
    #
    # Add it as all zero.
    # --------------------------------------------------------

    missing_in_current = (
        old_dummy_set
        - new_dummy_set
    )

    for col in sorted(
        missing_in_current
    ):
        current_dummies[col] = 0.0

    # --------------------------------------------------------
    # CURRENT CATEGORY NEVER SEEN IN TRAINING
    #
    # Drop it. The historical model cannot use it.
    # --------------------------------------------------------

    unseen_current = (
        new_dummy_set
        - old_dummy_set
    )

    if unseen_current:

        current_dummies.drop(
            columns=sorted(
                unseen_current
            ),
            inplace=True
        )

    # --------------------------------------------------------
    # FORCE EXACT HISTORICAL ORDER
    # --------------------------------------------------------

    current_dummies = (
        current_dummies[
            old_dummies
        ]
        .copy()
    )

    print(
        "\nDummy alignment"
    )

    print(
        "---------------"
    )

    print(
        f"Historical dummy columns: "
        f"{len(old_dummies)}"
    )

    print(
        f"Added missing current dummies: "
        f"{len(missing_in_current)}"
    )

    print(
        f"Dropped unseen current dummies: "
        f"{len(unseen_current)}"
    )

    return current_dummies


# ============================================================
# TRANSFORM CURRENT METRICS
# ============================================================

def transform_current_metrics(
    current_nocat,
    score_params
):

    current_nocat = (
        current_nocat.copy()
    )

    # --------------------------------------------------------
    # EXACT HISTORICAL FEATURE SET
    # --------------------------------------------------------

    historical_features = list(
        score_params.index
    )

    missing_features = [
        col
        for col in historical_features
        if col not in current_nocat.columns
    ]

    if missing_features:
        raise ValueError(
            "Current no-category data is "
            "missing historical features:\n"
            f"{missing_features}"
        )

    # Extra current columns must not affect the model.
    current_nocat = (
        current_nocat[
            historical_features
        ]
        .copy()
    )

    current_nocat = (
        current_nocat.apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    if current_nocat.isna().any().any():

        bad_cols = current_nocat.columns[
            current_nocat.isna().any()
        ].tolist()

        raise ValueError(
            "NaN/non-numeric values found "
            "in current metrics:\n"
            f"{bad_cols}"
        )

    # --------------------------------------------------------
    # APPLY HISTORICAL TRANSFORMS
    # --------------------------------------------------------

    skew_cols = (
        score_params.index[
            score_params[
                "skew_score"
            ].astype(bool)
        ]
    )

    fattail_cols = (
        score_params.index[
            score_params[
                "fattail_score"
            ].astype(bool)
        ]
    )

    for col in skew_cols:

        if (
            current_nocat[col] < 0
        ).any():
            raise ValueError(
                f'Negative values found in '
                f'log1p feature "{col}".'
            )

        current_nocat[col] = (
            np.log1p(
                current_nocat[col]
            )
        )

    for col in fattail_cols:

        current_nocat[col] = (
            np.arcsinh(
                current_nocat[col]
            )
        )

    # --------------------------------------------------------
    # SCALE WITH HISTORICAL MEAN / STD
    # --------------------------------------------------------

    mean_values = (
        pd.to_numeric(
            score_params[
                "mean"
            ],
            errors="raise"
        )
    )

    std_values = (
        pd.to_numeric(
            score_params[
                "std"
            ],
            errors="raise"
        )
    )

    if (
        std_values <= 0
    ).any():

        bad_cols = (
            std_values[
                std_values <= 0
            ]
            .index
            .tolist()
        )

        raise ValueError(
            "Invalid historical standard "
            "deviations for:\n"
            f"{bad_cols}"
        )

    scaled_data = (
        current_nocat
        - mean_values
    ) / std_values

    return scaled_data


# ============================================================
# GROUP CURRENT METRIC SCORES
# ============================================================

def group_current_data(
    scaled_data,
    load_mat_df
):

    missing_load_features = [
        col
        for col in load_mat_df.index
        if col not in scaled_data.columns
    ]

    if missing_load_features:
        raise ValueError(
            "Scaled current data is missing "
            "loading-matrix features:\n"
            f"{missing_load_features}"
        )

    scaled_data = (
        scaled_data[
            load_mat_df.index
        ]
    )

    grouped_values = (
        scaled_data.to_numpy()
        @
        load_mat_df.to_numpy()
    )

    grouped_data = pd.DataFrame(
        grouped_values,
        index=scaled_data.index,
        columns=load_mat_df.columns
    )

    return grouped_data


# ============================================================
# SAVE SEGMENT DATA
# ============================================================

def save_segment_data_wcats(
    current_data_grouped,
    current_raw_data,
    load_mat_df,
    data_set_path,
    categories
):

    # --------------------------------------------------------
    # GROUPED FEATURES:
    # loading columns with >1 contributing metric.
    # --------------------------------------------------------

    load_counts = (
        load_mat_df
        .astype(bool)
        .sum(
            axis=0
        )
    )

    group_cols = list(
        load_counts[
            load_counts > 1
        ].index
    )

    # --------------------------------------------------------
    # SINGLE-METRIC GROUPS
    # --------------------------------------------------------

    no_group_cols = list(
        load_counts[
            load_counts == 1
        ].index
    )

    grouped_part = (
        current_data_grouped[
            group_cols
        ]
        .copy()
    )

    # --------------------------------------------------------
    # For singleton group scores we normally use the grouped
    # score itself, because the load-matrix column name is a
    # group name rather than necessarily the raw metric name.
    # --------------------------------------------------------

    for col in no_group_cols:

        if col not in grouped_part.columns:

            grouped_part[col] = (
                current_data_grouped[
                    col
                ]
            )

    # --------------------------------------------------------
    # ADD REQUESTED RAW DEMOGRAPHIC CATEGORY COLUMNS
    # --------------------------------------------------------

    available_categories = [
        col
        for col in categories
        if col in current_raw_data.columns
    ]

    missing_categories = [
        col
        for col in categories
        if col not in current_raw_data.columns
    ]

    if missing_categories:

        print(
            "\nWarning: requested segment columns "
            "not found in current raw data:"
        )

        print(
            missing_categories
        )

    segment_df = (
        grouped_part.join(
            current_raw_data[
                available_categories
            ],
            how="left"
        )
    )

    segment_path = (
        data_set_path.replace(
            ".csv",
            "_current_groupmets_segment.csv"
        )
    )

    segment_df.to_csv(
        segment_path
    )

    print(
        f"\nSaved segmentation dataset:\n"
        f"{segment_path}"
    )

    return segment_df


# ============================================================
# MAIN — RESCORE CURRENT DATA WITH CATEGORIES
# ============================================================

def rescore_wcats(
    data_set_path,
    categories,
    groups
):

    # ========================================================
    # PATHS
    # ========================================================

    current_path = (
        data_set_path.replace(
            ".csv",
            "_current.csv"
        )
    )

    historical_nocat_path = (
        data_set_path.replace(
            ".csv",
            "_nocat.csv"
        )
    )

    current_nocat_path = (
        data_set_path.replace(
            ".csv",
            "_current_nocat.csv"
        )
    )

    historical_load_path = (
        historical_nocat_path.replace(
            ".csv",
            "_load_mat.csv"
        )
    )

    historical_score_path = (
        historical_nocat_path.replace(
            ".csv",
            "_score_params.csv"
        )
    )


    # ========================================================
    # FILE CHECKS
    # ========================================================

    required_files = [
        current_path,
        historical_load_path,
        historical_score_path,
        data_set_path.replace(
            ".csv",
            "_dummies_groupmets.csv"
        )
    ]

    missing_files = [
        path
        for path in required_files
        if not os.path.isfile(path)
    ]

    if missing_files:

        raise FileNotFoundError(
            "Required files are missing:\n"
            + "\n".join(
                missing_files
            )
        )


    # ========================================================
    # 1. CREATE CURRENT DEMOGRAPHIC DUMMIES
    # ========================================================

    (
        current_dummies,
        current_nocat,
        current_dummy_defs
    ) = dummy_variables(
        current_path=current_path,
        groups=groups
    )


    # ========================================================
    # 2. ALIGN CURRENT DUMMIES TO TRAINING SCHEMA
    # ========================================================

    current_dummies = (
        align_dummies(
            current_dummies,
            data_set_path
        )
    )


    # ========================================================
    # SAVE ALIGNED CURRENT DUMMIES
    # ========================================================

    aligned_dummy_path = (
        data_set_path.replace(
            ".csv",
            "_current_dummies_groupscore.csv"
        )
    )

    current_dummies.to_csv(
        aligned_dummy_path
    )

    print(
        f"\nSaved aligned current dummies:\n"
        f"{aligned_dummy_path}"
    )


    # ========================================================
    # 3. LOAD HISTORICAL TRANSFORM PARAMETERS
    # ========================================================

    load_mat_df = pd.read_csv(
        historical_load_path,
        index_col=0
    )

    score_df = pd.read_csv(
        historical_score_path,
        index_col=0
    )


    # ========================================================
    # 4. RELOAD CURRENT NUMERIC DATA
    # ========================================================

    if not os.path.isfile(
        current_nocat_path
    ):

        raise FileNotFoundError(
            "Current no-category dataset "
            "was not created:\n"
            f"{current_nocat_path}"
        )

    current_nocat = pd.read_csv(
        current_nocat_path,
        index_col=[0, 1]
    )


    # ========================================================
    # 5. TRANSFORM + SCALE CURRENT METRICS
    # ========================================================

    scaled_data = (
        transform_current_metrics(
            current_nocat=current_nocat,
            score_params=score_df
        )
    )

    current_scores_path = (
        data_set_path.replace(
            ".csv",
            "_current_scores.csv"
        )
    )

    scaled_data.to_csv(
        current_scores_path
    )

    print(
        f"\nSaved current standardized scores:\n"
        f"{current_scores_path}"
    )


    # ========================================================
    # 6. APPLY HISTORICAL METRIC GROUPS
    # ========================================================

    grouped_data = (
        group_current_data(
            scaled_data=scaled_data,
            load_mat_df=load_mat_df
        )
    )


    # ========================================================
    # 7. MERGE GROUPED METRICS + ALIGNED DUMMIES
    # ========================================================

    if not grouped_data.index.equals(
        current_dummies.index
    ):

        missing_dummies = (
            grouped_data.index.difference(
                current_dummies.index
            )
        )

        missing_groups = (
            current_dummies.index.difference(
                grouped_data.index
            )
        )

        if (
            len(missing_dummies) > 0
            or len(missing_groups) > 0
        ):

            raise ValueError(
                "Current grouped metrics and "
                "dummy datasets have different indices.\n"
                f"Missing in dummies: "
                f"{len(missing_dummies):,}\n"
                f"Missing in grouped data: "
                f"{len(missing_groups):,}"
            )

        current_dummies = (
            current_dummies.reindex(
                grouped_data.index
            )
        )

    group_dum_df = (
        grouped_data.merge(
            current_dummies,
            left_index=True,
            right_index=True,
            how="inner",
            validate="one_to_one"
        )
    )


    # ========================================================
    # 8. SAVE FINAL CURRENT MODEL DATASET
    # ========================================================

    current_groupscore_path = (
        data_set_path.replace(
            ".csv",
            "_current_groupscore.csv"
        )
    )

    group_dum_df.to_csv(
        current_groupscore_path
    )

    print(
        f"\nSaved current grouped metrics + "
        f"demographic dummies:\n"
        f"{current_groupscore_path}"
    )


    # ========================================================
    # 9. SEGMENTATION DATA
    # ========================================================

    current_raw_data = pd.read_csv(
        current_path,
        index_col=[0, 1]
    )

    segment_df = (
        save_segment_data_wcats(
            current_data_grouped=grouped_data,
            current_raw_data=current_raw_data,
            load_mat_df=load_mat_df,
            data_set_path=data_set_path,
            categories=categories
        )
    )


    # ========================================================
    # QA
    # ========================================================

    print(
        "\nCurrent demographic rescoring QA"
    )

    print(
        "--------------------------------"
    )

    print(
        f"Current accounts: "
        f"{len(group_dum_df):,}"
    )

    print(
        f"Grouped behavioral features: "
        f"{grouped_data.shape[1]}"
    )

    print(
        f"Demographic dummy features: "
        f"{current_dummies.shape[1]}"
    )

    print(
        f"Final model features: "
        f"{group_dum_df.shape[1]}"
    )

    print(
        "\nCurrent demographic rescoring complete."
    )

    return (
        group_dum_df,
        grouped_data,
        current_dummies,
        segment_df
    )


# ============================================================
# SAME COUNTRY GROUPING USED FOR HISTORICAL TRAINING
# ============================================================

groups = {

    "country": {

        "North America": [
            "US",
            "CA"
        ],

        "Western Europe": [
            "GB",
            "DE",
            "FR"
        ],

        "Asia": [
            "JP"
        ]
    }
}


# ============================================================
# DEMOGRAPHIC FIELDS TO KEEP FOR SEGMENT ANALYSIS
# ============================================================

categories = [
    "channel",
    "country",
    "customer_age"
]


# ============================================================
# RUN
# ============================================================

(
    current_demographic_groupscore,
    current_behavior_groupscore,
    current_demographic_dummies,
    current_segment_data
) = rescore_wcats(
    data_set_path=data_set_path,
    categories=categories,
    groups=groups
)


# ============================================================
# DISPLAY FINAL MODEL DATA
# ============================================================

print(
    "\nFinal current model dataset:"
)

display(
    current_demographic_groupscore.head()
)


print(
    "\nCurrent segmentation dataset:"
)

display(
    current_segment_data.head()
)

Loaded current rows: 11,832
Saved current dummy dataset:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_dummies_groupscore.csv
Saved current no-category dataset:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_nocat.csv
Current dummy columns: 29

Dummy alignment
---------------
Historical dummy columns: 29
Added missing current dummies: 0
Dropped unseen current dummies: 0

Saved aligned current dummies:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_dummies_groupscore.csv

Saved current standardized scores:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_scores.csv

Saved current grouped metrics + demographic dummies:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_groupscore.csv

Saved segmentation dataset:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_demographic_current_groupmets_

,,group_1,group_2,group_3,group_4,channel_appstore1,channel_appstore2,channel_nan,channel_web,country_group_-na-,country_group_AR,...,country_group_MX,country_group_NL,country_group_NO,country_group_NZ,country_group_North America,country_group_PT,country_group_RU,country_group_SE,country_group_Western Europe,country_group_nan
account_id,observation_date,,,,,,,,,,,,,,,,,,,,,
0,2020-05-10 00:00:00,0.764752,1.396817,-0.289952,1.915891,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2020-05-10 00:00:00,0.292810,0.875108,-0.161488,1.816383,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2020-05-10 00:00:00,-0.379744,-0.170617,-0.289952,2.148076,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2020-05-10 00:00:00,-0.097911,-0.099284,0.866220,1.915891,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,2020-05-10 00:00:00,1.788583,-0.128712,1.251611,1.285673,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0



Current segmentation dataset:


,,group_1,group_2,group_3,group_4,channel,country,customer_age
account_id,observation_date,,,,,,,
0,2020-05-10 00:00:00,0.764752,1.396817,-0.289952,1.915891,appstore2,NaN,75.142466
1,2020-05-10 00:00:00,0.292810,0.875108,-0.161488,1.816383,appstore1,AU,64.602740
2,2020-05-10 00:00:00,-0.379744,-0.170617,-0.289952,2.148076,appstore2,CN,43.476712
3,2020-05-10 00:00:00,-0.097911,-0.099284,0.866220,1.915891,appstore2,NaN,44.917808
4,2020-05-10 00:00:00,1.788583,-0.128712,1.251611,1.285673,appstore1,US,44.323288
